In [17]:
import os
import json, re
import zipfile
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

In [5]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:4]}")
else:
    print("GROQ API Key not set")

MODEL = "openai/gpt-oss-120b"
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

GROQ API Key exists and begins gsk_


In [6]:
def build_schema_prompt(application):
    return f"""
You are a database architect.

Create a realistic relational database schema for:

{application}

Requirements:
- 4 to 6 tables
- Include primary keys
- Include foreign keys
- Use realistic column names

Return ONLY valid JSON.

{{
  "application":"",
  "tables":[
    {{
      "table_name":"",
      "description":"",
      "columns":[
        {{
          "name":"",
          "type":"",
          "primary_key":false,
          "foreign_key":""
        }}
      ]
    }}
  ]
}}
"""

In [ ]:
def build_table_prompt(
    table_schema,
    rows
):
    return f"""
Generate {rows} realistic records.

Table Schema:

{json.dumps(table_schema, indent=2)}

Rules:
- Return ONLY JSON array.
- Generate realistic values.
- No markdown.
- No explanation.

Example:

[
  {{
    "id": 1
  }}
]
"""

In [8]:
def generate_groq(
    prompt,
    temperature=0.3
):
    response = groq.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [10]:
def extract_json(text):

    match = re.search(
        r'(\{.*\}|\[.*\])',
        text,
        re.DOTALL
    )

    if not match:
        raise ValueError(
            "JSON not found"
        )

    return json.loads(
        match.group()
    )

In [11]:
def generate_schema(
    application
):

    prompt = build_schema_prompt(
        application
    )

    response = generate_groq(
        prompt
    )

    return extract_json(
        response
    )

In [14]:
def generate_dataset(
    schema,
    rows
):

    tables = {}

    for table in schema["tables"]:

        prompt = build_table_prompt(
            table,
            rows
        )

        response = generate_groq(
            prompt,
            temperature=0.7
        )

        data = extract_json(
            response
        )

        tables[
            table["table_name"]
        ] = pd.DataFrame(data)

    return tables

In [15]:
def schema_to_markdown(
    schema
):

    md = f"# {schema['application']}\n\n"

    for table in schema["tables"]:

        md += (
            f"## {table['table_name']}\n\n"
        )

        md += "| Column | Type |\n"
        md += "|--------|------|\n"

        for col in table["columns"]:

            md += (
                f"| {col['name']} "
                f"| {col['type']} |\n"
            )

        md += "\n"

    return md

In [16]:
def preview_table(
    tables
):

    first_table = next(
        iter(tables.values())
    )

    return first_table.head(10)

In [18]:
def create_zip(
    tables
):

    temp_dir = tempfile.mkdtemp()

    for name, df in tables.items():

        df.to_csv(
            os.path.join(
                temp_dir,
                f"{name}.csv"
            ),
            index=False
        )

    zip_path = os.path.join(
        temp_dir,
        "dataset.zip"
    )

    with zipfile.ZipFile(
        zip_path,
        "w"
    ) as z:

        for name in tables:

            z.write(
                os.path.join(
                    temp_dir,
                    f"{name}.csv"
                ),
                f"{name}.csv"
            )

    return zip_path

In [28]:
def generate_schema_ui(application):

    schema = generate_schema(application)

    markdown = schema_to_markdown(schema)

    return markdown, schema

In [36]:
def generate_database_ui(
    application,
    rows
):

    # Step 1
    schema = generate_schema(
        application
    )

    # Step 2
    markdown = schema_to_markdown(
        schema
    )

    # Step 3
    tables = generate_dataset(
        schema,
        rows
    )

    # Step 4
    zip_path = create_zip(
        tables
    )

    # Step 5
    first_table = next(
        iter(tables.values())
    )

    status = (
        f"Generated "
        f"{len(tables)} tables "
        f"with {rows} rows each"
    )

    return (
        markdown,
        first_table.head(10),
        zip_path,
        schema,
        tables,
        status
    )

In [37]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 🚀 Synthetic Database Generator

        Generate realistic database schemas and synthetic datasets
        for any application domain using GPT-OSS-120B.
        """
    )

    schema_state = gr.State(None)
    dataset_state = gr.State(None)

    with gr.Row():

        application = gr.Dropdown(
            choices=[
                "E-Commerce",
                "Healthcare",
                "Banking",
                "School Management",
                "Hospital",
                "Hotel Booking",
                "Food Delivery",
                "Library Management",
                "HR Management"
            ],
            value="E-Commerce",
            label="Application Type",
            allow_custom_value=True
        )

        rows = gr.Slider(
            minimum=10,
            maximum=100,
            value=20,
            step=10,
            label="Rows Per Table"
        )

    with gr.Row():

        generate_schema_btn = gr.Button(
            "Generate Schema",
            variant="primary"
        )

        generate_database_btn = gr.Button(
            "Generate Database",
            variant="secondary"
        )

    with gr.Tabs():

        with gr.Tab("Schema"):

            schema_view = gr.Markdown(
                value="Schema will appear here..."
            )

        with gr.Tab("Dataset Preview"):

            dataset_preview = gr.Dataframe(
                interactive=False
            )

        with gr.Tab("Download"):

            download_file = gr.File(
                label="Download Dataset ZIP"
            )

    with gr.Accordion(
        "Generation Details",
        open=False
    ):

        generation_status = gr.Textbox(
            label="Status",
            interactive=False
        )

    # -----------------------------
    # Button Wiring
    # -----------------------------

    generate_schema_btn.click(
        fn=generate_schema_ui,
        inputs=[application],
        outputs=[
            schema_view,
            schema_state
        ]
    )

    generate_database_btn.click(
    fn=generate_database_ui,
    inputs=[
        application,
        rows
    ],
    outputs=[
        schema_view,
        dataset_preview,
        download_file,
        schema_state,
        dataset_state,
        generation_status
    ]
)

In [ ]:
demo.launch(
    theme=gr.themes.Soft()
)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [24]:
print(gr.__version__)

6.14.0
